In [1]:
import numpy as np
import pandas as pd
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
import optuna
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Import custom functions
import sys
sys.path.append('../files')
from functions import dataSetup
from CONSTANTS import COIN

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"Loading and preprocessing {COIN} data...")

Loading and preprocessing BTC data...


In [2]:
# Load and preprocess data
data = pd.read_csv(f'../data/{COIN}_df.csv')
data = data.loc[:, ~data.columns.str.contains('^Unnamed:')]
data = dataSetup(data, trainingColPath='../files/training_columns.txt')

print(f"Data shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
print(f"Columns: {list(data.columns)}")
data.head()


Data shape: (3686, 13)
Date range: 2015-07-20 00:00:00 to 2025-08-21 00:00:00
Columns: ['BB_Lower', 'BB_Upper', 'value', 'avg_sentiment', 'volume', 'OBV', 'SMA_7', 'Volume_MA_7', 'low', 'high', 'open', 'close', 'gradient']


,BB_Lower,BB_Upper,value,avg_sentiment,volume,OBV,SMA_7,Volume_MA_7,low,high,open,close,gradient
time,,,,,,,,,,,,,
2015-07-20,268.225208,299.188792,50.0,0.0,782.883420,-3.748088e+06,283.615714,4417.348637,277.37,280.00,277.98,280.00,0.00
2015-07-21,266.070954,300.106046,50.0,0.0,4943.559434,-3.748871e+06,285.645714,5228.512882,276.85,281.27,279.96,277.32,-2.68
2015-07-22,263.804410,301.155590,50.0,0.0,4687.909383,-3.743928e+06,288.280000,5345.414337,275.01,278.54,277.33,277.89,0.57
2015-07-23,263.024045,301.345955,50.0,0.0,5306.919575,-3.748616e+06,290.037143,5421.928306,276.28,279.75,277.96,277.39,-0.50
2015-07-24,261.412051,301.912949,50.0,0.0,7362.469083,-3.743309e+06,291.622857,5397.937160,276.43,291.52,277.23,289.12,11.73


In [3]:
# Temporal Fusion Transformer Implementation
class VariableSelectionNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_rate=0.1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.grn = GatedResidualNetwork(input_size, hidden_size, output_size=input_size, dropout_rate=dropout_rate)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        sparse_weights = self.grn(x)
        sparse_weights = self.softmax(sparse_weights)

        # Apply variable selection
        selected = x * sparse_weights
        return selected, sparse_weights

class GatedResidualNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, dropout_rate=0.1):
        super().__init__()
        if output_size is None:
            output_size = input_size

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_rate)
        self.gate = nn.Linear(hidden_size, output_size)
        self.layer_norm = nn.LayerNorm(output_size)

        # Skip connection
        if input_size != output_size:
            self.skip_connection = nn.Linear(input_size, output_size)
        else:
            self.skip_connection = None

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        residual = x

        # Apply first linear layer and activation
        x = torch.relu(self.linear1(x))
        x = self.dropout(x)

        # Apply second linear layer
        x = self.linear2(x)

        # Gating mechanism
        gate = torch.sigmoid(self.gate(torch.relu(self.linear1(residual))))
        x = x * gate

        # Skip connection
        if self.skip_connection is not None:
            residual = self.skip_connection(residual)

        # Add residual and apply layer norm
        x = self.layer_norm(x + residual)

        return x

class InterpretableMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout_rate)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, query, key, value, mask=None):
        batch_size, seq_len = query.size(0), query.size(1)
        residual = query

        # Linear projections
        Q = self.w_q(query).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = torch.matmul(attention_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # Output projection and residual connection
        output = self.w_o(context)
        output = self.layer_norm(output + residual)

        return output, attention_weights.mean(dim=1)  # Average attention across heads

class TemporalFusionTransformer(nn.Module):
    def __init__(self,
                 input_size,
                 hidden_size=128,
                 n_heads=4,
                 n_layers=2,
                 dropout_rate=0.1,
                 sequence_length=30,
                 prediction_horizon=1):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.sequence_length = sequence_length
        self.prediction_horizon = prediction_horizon

        # Input projection to hidden_size
        self.input_projection = nn.Linear(input_size, hidden_size)

        # Variable selection networks
        self.static_vsn = VariableSelectionNetwork(input_size, input_size, dropout_rate)
        self.temporal_vsn = VariableSelectionNetwork(input_size, input_size, dropout_rate)

        # Encoder
        self.encoder_layers = nn.ModuleList([
            InterpretableMultiHeadAttention(hidden_size, n_heads, dropout_rate)
            for _ in range(n_layers)
        ])

        # Decoder
        self.decoder_layers = nn.ModuleList([
            InterpretableMultiHeadAttention(hidden_size, n_heads, dropout_rate)
            for _ in range(n_layers)
        ])

        # GRN layers
        self.encoder_grn = GatedResidualNetwork(hidden_size, hidden_size, dropout_rate=dropout_rate)
        self.decoder_grn = GatedResidualNetwork(hidden_size, hidden_size, dropout_rate=dropout_rate)

        # Output layer
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, prediction_horizon)
        )

        # Positional encoding
        self.positional_encoding = self._create_positional_encoding(sequence_length, hidden_size)

    def _create_positional_encoding(self, seq_len, d_model):
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len).unsqueeze(1).float()

        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           -(np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        batch_size, seq_len = x.size(0), x.size(1)

        # Variable selection
        selected_features, vsn_weights = self.temporal_vsn(x)

        # Project to hidden dimension
        projected_features = self.input_projection(selected_features)

        # Add positional encoding
        pos_encoding = self.positional_encoding[:, :seq_len, :].to(x.device)
        encoder_input = projected_features + pos_encoding

        # Encoder
        encoder_output = encoder_input
        encoder_attention_weights = []
        for layer in self.encoder_layers:
            encoder_output, attn_weights = layer(encoder_output, encoder_output, encoder_output)
            encoder_attention_weights.append(attn_weights)

        encoder_output = self.encoder_grn(encoder_output)

        # Use the last timestep for prediction
        final_representation = encoder_output[:, -1, :]  # (batch_size, hidden_size)

        # Output prediction
        output = self.output_layer(final_representation)

        return output, {
            'vsn_weights': vsn_weights,
            'encoder_attention': encoder_attention_weights
        }


In [4]:
# Data preprocessing and normalization
def create_sequences(data, sequence_length=30, prediction_horizon=1):
    """Create sequences for time series prediction"""
    X, y = [], []

    for i in range(sequence_length, len(data) - prediction_horizon + 1):
        X.append(data[i-sequence_length:i])
        y.append(data[i:i+prediction_horizon, -1])  # Predict 'close' price

    return np.array(X), np.array(y)

def normalize_data(data, method='standard', target_col='close'):
    """Normalize data using different methods"""
    data_copy = data.copy()

    # Separate target column
    target_data = data_copy[target_col].values.reshape(-1, 1)
    feature_data = data_copy.drop(columns=[target_col]).values

    if method == 'standard':
        feature_scaler = StandardScaler()
        target_scaler = StandardScaler()
    elif method == 'minmax':
        feature_scaler = MinMaxScaler()
        target_scaler = MinMaxScaler()
    elif method == 'robust':
        feature_scaler = RobustScaler()
        target_scaler = RobustScaler()
    elif method == 'log':
        # Log transform (add small constant to avoid log(0))
        feature_data = np.log(feature_data + 1e-8)
        target_data = np.log(target_data + 1e-8)
        feature_scaler = StandardScaler()
        target_scaler = StandardScaler()
    else:
        raise ValueError("Method must be 'standard', 'minmax', 'robust', or 'log'")

    # Fit and transform
    feature_data_scaled = feature_scaler.fit_transform(feature_data)
    target_data_scaled = target_scaler.fit_transform(target_data)

    # Combine back
    scaled_data = np.column_stack([feature_data_scaled, target_data_scaled.flatten()])

    return scaled_data, feature_scaler, target_scaler

# Test different normalization methods
normalization_methods = ['standard', 'minmax', 'robust', 'log']
normalization_results = {}

print("Testing different normalization methods...")

for method in normalization_methods:
    try:
        # Prepare data
        data_clean = data.dropna()

        # Normalize
        normalized_data, feature_scaler, target_scaler = normalize_data(data_clean, method=method)

        print(f"✓ {method.capitalize()} normalization successful")
        normalization_results[method] = {
            'data': normalized_data,
            'feature_scaler': feature_scaler,
            'target_scaler': target_scaler,
            'original_data': data_clean
        }
    except Exception as e:
        print(f"✗ {method.capitalize()} normalization failed: {str(e)}")


Testing different normalization methods...
✓ Standard normalization successful
✓ Minmax normalization successful
✓ Robust normalization successful
✓ Log normalization successful


In [5]:
# Hyperparameter tuning setup
def objective(trial):
    """Optuna objective function for hyperparameter tuning"""

    # Suggest hyperparameters - MINI GRID FOR LOCAL EXECUTION
    hidden_size = trial.suggest_categorical('hidden_size', [64, 128])  # Reduced from [64, 128, 256]
    n_heads = trial.suggest_categorical('n_heads', [2, 4])  # Reduced from [2, 4, 8]
    n_layers = trial.suggest_int('n_layers', 1, 2)  # Reduced from 1-4 to 1-2
    dropout_rate = trial.suggest_categorical('dropout_rate', [0.1, 0.2, 0.3])  # Fixed options instead of float range
    learning_rate = trial.suggest_categorical('learning_rate', [1e-4, 1e-3])  # Fixed options instead of log-uniform
    batch_size = trial.suggest_categorical('batch_size', [16, 32])  # Reduced from [16, 32, 64]
    sequence_length = trial.suggest_categorical('sequence_length', [20, 30, 40])  # Fixed options instead of range

    # Choose normalization method - keep all to find best
    norm_method = trial.suggest_categorical('normalization', list(normalization_results.keys()))

    try:
        # Get normalized data
        norm_data = normalization_results[norm_method]['data']
        target_scaler = normalization_results[norm_method]['target_scaler']

        # Create sequences
        X, y = create_sequences(norm_data, sequence_length=sequence_length)

        # Split data
        train_size = int(0.8 * len(X))
        X_train, X_val = X[:train_size], X[train_size:]
        y_train, y_val = y[:train_size], y[train_size:]

        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train)
        y_train_tensor = torch.FloatTensor(y_train)
        X_val_tensor = torch.FloatTensor(X_val)
        y_val_tensor = torch.FloatTensor(y_val)

        # Create data loaders
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Initialize model
        model = TemporalFusionTransformer(
            input_size=X_train.shape[2],
            hidden_size=hidden_size,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout_rate=dropout_rate,
            sequence_length=sequence_length
        )

        # Training setup
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        # Training loop - REDUCED EPOCHS FOR FASTER LOCAL EXECUTION
        model.train()
        for epoch in range(5):  # Reduced from 10 to 5 epochs for faster tuning
            train_loss = 0.0
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                predictions, _ = model(batch_X)
                loss = criterion(predictions, batch_y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

        # Validation
        model.eval()
        val_predictions = []
        val_targets = []

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                predictions, _ = model(batch_X)
                val_predictions.extend(predictions.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())

        # Calculate RMSE
        val_predictions = np.array(val_predictions)
        val_targets = np.array(val_targets)

        # Back-transform predictions
        val_predictions_original = target_scaler.inverse_transform(val_predictions.reshape(-1, 1))
        val_targets_original = target_scaler.inverse_transform(val_targets.reshape(-1, 1))

        rmse = np.sqrt(mean_squared_error(val_targets_original, val_predictions_original))

        return rmse

    except Exception as e:
        return float('inf')  # Return large value for failed trials

print("Starting hyperparameter tuning with mini grid for local execution...")
print("Grid size: 2×2×2×3×2×2×3×4 = 1,152 possible combinations")
print("Testing subset with reduced trials for 5-minute execution...")

# Run hyperparameter optimization with VERY REDUCED TRIALS for 5-minute execution
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=8, timeout=300)  # Reduced to 8 trials, 5 min timeout

print(f"Best RMSE: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

# Print optimization summary
print(f"\nOptimization completed!")
print(f"Total trials: {len(study.trials)}")
print(f"Best trial number: {study.best_trial.number}")
print(f"Grid reduction: Using focused mini-grid for 5-minute local execution")


[I 2025-08-29 19:25:37,932] A new study created in memory with name: no-name-60073656-9f58-41af-8907-15fd927e0d8e


Starting hyperparameter tuning with mini grid for local execution...
Grid size: 2×2×2×3×2×2×3×4 = 1,152 possible combinations
Testing subset with reduced trials for 5-minute execution...


[I 2025-08-29 19:25:41,376] Trial 0 finished with value: 30766.55859375 and parameters: {'hidden_size': 128, 'n_heads': 2, 'n_layers': 1, 'dropout_rate': 0.3, 'learning_rate': 0.0001, 'batch_size': 32, 'sequence_length': 20, 'normalization': 'standard'}. Best is trial 0 with value: 30766.55859375.
[I 2025-08-29 19:25:45,584] Trial 1 finished with value: 43794.41015625 and parameters: {'hidden_size': 64, 'n_heads': 2, 'n_layers': 2, 'dropout_rate': 0.1, 'learning_rate': 0.001, 'batch_size': 16, 'sequence_length': 20, 'normalization': 'robust'}. Best is trial 0 with value: 30766.55859375.
[I 2025-08-29 19:25:49,037] Trial 2 finished with value: 51031.125 and parameters: {'hidden_size': 64, 'n_heads': 2, 'n_layers': 2, 'dropout_rate': 0.3, 'learning_rate': 0.0001, 'batch_size': 32, 'sequence_length': 30, 'normalization': 'minmax'}. Best is trial 0 with value: 30766.55859375.
[I 2025-08-29 19:25:52,768] Trial 3 finished with value: 36460.4609375 and parameters: {'hidden_size': 128, 'n_head

Best RMSE: 24380.5781
Best parameters: {'hidden_size': 128, 'n_heads': 4, 'n_layers': 2, 'dropout_rate': 0.1, 'learning_rate': 0.0001, 'batch_size': 16, 'sequence_length': 30, 'normalization': 'standard'}

Optimization completed!
Total trials: 8
Best trial number: 4
Grid reduction: Using focused mini-grid for 5-minute local execution


In [6]:
# Train final model with best parameters
best_params = study.best_params
norm_method = best_params['normalization']

print(f"Training final model with best parameters...")
print(f"Normalization method: {norm_method}")

# Get the best normalized data
best_norm_data = normalization_results[norm_method]['data']
best_feature_scaler = normalization_results[norm_method]['feature_scaler']
best_target_scaler = normalization_results[norm_method]['target_scaler']
original_data = normalization_results[norm_method]['original_data']

# Create sequences with best parameters
X, y = create_sequences(best_norm_data, sequence_length=best_params['sequence_length'])

# Split data
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False)

# Initialize final model
final_model = TemporalFusionTransformer(
    input_size=X_train.shape[2],
    hidden_size=best_params['hidden_size'],
    n_heads=best_params['n_heads'],
    n_layers=best_params['n_layers'],
    dropout_rate=best_params['dropout_rate'],
    sequence_length=best_params['sequence_length']
)

# Training setup
criterion = nn.MSELoss()
optimizer = optim.Adam(final_model.parameters(), lr=best_params['learning_rate'])

# Training loop with early stopping
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 10
patience_counter = 0

print("Training final model...")

for epoch in range(100):
    # Training
    final_model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions, interpretability = final_model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # Validation
    final_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            predictions, _ = final_model(batch_X)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()

    val_loss /= len(test_loader)
    val_losses.append(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(final_model.state_dict(), f'../models/{COIN}/{COIN}_tft_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")

print("Training completed!")


Training final model with best parameters...
Normalization method: standard
Training final model...
Epoch 10: Train Loss = 0.007017, Val Loss = 0.470779
Epoch 20: Train Loss = 0.005154, Val Loss = 0.419173
Epoch 30: Train Loss = 0.004053, Val Loss = 0.389355
Early stopping at epoch 34
Training completed!


In [7]:
# Load best model for predictions
final_model.load_state_dict(torch.load(f'../models/{COIN}/{COIN}_tft_model.pth'))
final_model.eval()

# Make predictions on test set
test_predictions = []
test_targets = []
attention_weights_list = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        predictions, interpretability = final_model(batch_X)
        test_predictions.extend(predictions.cpu().numpy())
        test_targets.extend(batch_y.cpu().numpy())

        # Store attention weights for interpretability
        if 'encoder_attention' in interpretability:
            attention_weights_list.append(interpretability['encoder_attention'][0].cpu().numpy())

# Convert to arrays
test_predictions = np.array(test_predictions)
test_targets = np.array(test_targets)

# Back-transform predictions to original scale
test_predictions_original = best_target_scaler.inverse_transform(test_predictions.reshape(-1, 1))
test_targets_original = best_target_scaler.inverse_transform(test_targets.reshape(-1, 1))

# Calculate final RMSE
final_rmse = np.sqrt(mean_squared_error(test_targets_original, test_predictions_original))
print(f"Final Test RMSE: {final_rmse:.4f}")

# Create prediction dates for test set
test_start_idx = train_size + best_params['sequence_length']
test_prediction_dates = original_data.index[test_start_idx:test_start_idx + len(test_predictions_original)]

# Create test predictions DataFrame
test_predictions_df = pd.DataFrame({
    'predicted_price': test_predictions_original.flatten()
}, index=test_prediction_dates)

print(f"Test predictions shape: {test_predictions_df.shape}")
print(f"Test date range: {test_predictions_df.index.min()} to {test_predictions_df.index.max()}")

# GENERATE FUTURE PREDICTIONS FOR NEXT 7 DAYS
print("\nGenerating future predictions for next 7 days...")

def predict_future_days(model, data, scaler, feature_scaler, sequence_length, n_days=7):
    """
    Predict future prices for the next n_days using autoregressive approach
    """
    model.eval()

    # Use the last sequence_length days as input
    last_sequence = data[-sequence_length:].copy()

    future_predictions = []
    current_sequence = last_sequence.copy()

    with torch.no_grad():
        for day in range(n_days):
            # Prepare input tensor
            input_tensor = torch.FloatTensor(current_sequence).unsqueeze(0)  # Add batch dimension

            # Make prediction
            prediction, _ = model(input_tensor)
            prediction_value = prediction.cpu().numpy()[0, 0]  # Get scalar prediction

            # Back-transform to original scale
            prediction_original = scaler.inverse_transform([[prediction_value]])[0, 0]
            future_predictions.append(prediction_original)

            # Update sequence for next prediction (autoregressive)
            # Create new row with the predicted price and estimated other features
            new_row = current_sequence[-1].copy()  # Copy last row
            new_row[-1] = prediction_value  # Update close price (target is last column)

            # For other features, we'll use simple forward-filling or trend continuation
            # This is a simplification - in practice, you might want more sophisticated feature prediction
            for i in range(len(new_row) - 1):  # All features except the target
                # Simple trend continuation: use the change from previous day
                if len(current_sequence) >= 2:
                    change = current_sequence[-1, i] - current_sequence[-2, i]
                    new_row[i] = current_sequence[-1, i] + change * 0.1  # Damped trend
                else:
                    new_row[i] = current_sequence[-1, i]  # No change

            # Add new row to sequence and remove oldest
            current_sequence = np.vstack([current_sequence[1:], new_row])

    return np.array(future_predictions)

# Generate future predictions
future_prices = predict_future_days(
    final_model,
    best_norm_data,
    best_target_scaler,
    best_feature_scaler,
    best_params['sequence_length'],
    n_days=7
)

# Create future dates (next 7 days after the last date in data)
last_date = original_data.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7, freq='D')

# Create future predictions DataFrame
future_predictions_df = pd.DataFrame({
    'predicted_price': future_prices.flatten()
}, index=future_dates)

print(f"Future predictions shape: {future_predictions_df.shape}")
print(f"Future date range: {future_predictions_df.index.min()} to {future_predictions_df.index.max()}")

# Combine test predictions and future predictions
all_predictions_df = pd.concat([test_predictions_df, future_predictions_df])

# Save all predictions
all_predictions_df.to_csv(f'../predictions/{COIN}/tft_predictions.csv')
future_predictions_df.to_csv(f'../predictions/{COIN}/tft_future_7_days.csv')

print(f"All predictions saved to ../predictions/{COIN}/tft_predictions.csv")
print(f"Future 7-day predictions saved to ../predictions/{COIN}/tft_future_7_days.csv")

# Display future predictions
print(f"\n=== FUTURE 7-DAY PREDICTIONS ===")
for i, (date, price) in enumerate(future_predictions_df.iterrows(), 1):
    print(f"Day {i} ({date.strftime('%Y-%m-%d')}): ${price['predicted_price']:.2f}")

# Display test predictions sample
print(f"\nTest predictions sample:")
test_predictions_df.head(10)


Final Test RMSE: 17451.9043
Test predictions shape: (732, 1)
Test date range: 2023-08-21 00:00:00 to 2025-08-21 00:00:00

Generating future predictions for next 7 days...
Future predictions shape: (7, 1)
Future date range: 2025-08-22 00:00:00 to 2025-08-28 00:00:00
All predictions saved to ../predictions/BTC/tft_predictions.csv
Future 7-day predictions saved to ../predictions/BTC/tft_future_7_days.csv

=== FUTURE 7-DAY PREDICTIONS ===
Day 1 (2025-08-22): $76862.17
Day 2 (2025-08-23): $76824.28
Day 3 (2025-08-24): $76806.00
Day 4 (2025-08-25): $76789.40
Day 5 (2025-08-26): $76767.12
Day 6 (2025-08-27): $76744.86
Day 7 (2025-08-28): $76732.96

Test predictions sample:


,predicted_price
time,
2023-08-21,25859.003906
2023-08-22,26085.341797
2023-08-23,25951.943359
2023-08-24,25920.822266
2023-08-25,26280.962891
2023-08-26,26290.128906
2023-08-27,25641.191406
2023-08-28,25464.175781
2023-08-29,25665.509766


In [8]:
# calculate rmse for test predictions
test_rmse = np.sqrt(mean_squared_error(test_targets_original, test_predictions_original))
print(f"Test RMSE = ${test_rmse:.2f}")

Test RMSE = $17451.90


# Pipeline compliance additions

In [9]:
# 1) Save standardized (normalized-scale) RMSE for the validation/test split
#    Uses test_targets and test_predictions before inverse transform
try:
    from sklearn.metrics import mean_squared_error as _msq
    import numpy as _np
    import os as _os
    _os.makedirs(f'../predictions/{COIN}', exist_ok=True)
    _standardized_rmse = float(_np.sqrt(_msq(_np.array(test_targets).flatten(), _np.array(test_predictions).flatten())))
    _rmse_path = f'../predictions/{COIN}/tft_standardized_rmse.txt'
    with open(_rmse_path, 'w') as _f:
        _f.write(f"{_standardized_rmse:.6f}\n")
    print(f"Standardized RMSE saved to: {_rmse_path}")
except Exception as e:
    print('Failed to write standardized RMSE:', e)


Standardized RMSE saved to: ../predictions/BTC/tft_standardized_rmse.txt


In [10]:
# 2) Ensure 7-day forecast is saved as a DataFrame with date index and a single column 'predicted_price'
try:
    # Make sure index is named 'date' and save with index
    _fp = future_predictions_df[['predicted_price']].copy()
    _fp.index.name = 'date'
    _save_path = f'../predictions/{COIN}/tft_future_7_days.csv'
    _fp.to_csv(_save_path, index=True)
    print(f"Indexed 7-day forecast saved to: {_save_path}")
except Exception as e:
    print('Failed to save indexed 7-day forecast:', e)


Indexed 7-day forecast saved to: ../predictions/BTC/tft_future_7_days.csv


# Model improvement notes (non-executable)

- Predict 7 steps jointly by setting prediction_horizon=7 and training the decoder accordingly (reduces error accumulation vs autoregressive).
- Add known future covariates (calendar features like day-of-week, month, holiday flags) and time-varying re-computable features; pass them through temporal VSN.
- Use time-series cross-validation (rolling/blocked) for tuning instead of a single split; keep trials small for <5 min runs.
- Try cosine LR schedule with warmup and smaller learning rates; also test AdamW and weight decay.
- Increase sequence_length moderately (e.g., 60–120) if memory allows; balance with n_heads so hidden_size % n_heads == 0.
- Use quantile loss for probabilistic TFT and ensembling across seeds; average predictions.
- Improve feature scaling: fit scalers on train only and persist per normalization method; avoid leakage.
- Reduce drift in autoregressive future features: forecast key covariates or clamp to realistic ranges instead of naive propagation.
